In [ ]:
!uv pip install -U requests python-dotenv openai

PR_JSON_FILE = "_prs.json"
PR_TOPICS_FILE = "_topics.json"
MODEL = "gpt-4.1"  # GPT-5 is very slow

Using Python 3.12.11 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Resolved 20 packages in 204ms                                        
Audited 20 packages in 0.04ms


In [ ]:
# Get all PRs (title, date)
# Please set `GITHUB_TOKEN` to increate GitHub rate limit

import os
import requests
import json


url = "https://api.github.com/repos/materialsproject/pymatgen/pulls"
params = {"state": "all", "per_page": 100, "page": 1}

# Setup GitHub token (for higher API rate limit)
if not (token := os.environ.get("GITHUB_TOKEN")):
    raise RuntimeError("Please set your GitHub token in GITHUB_TOKEN env var.")
headers = {"Authorization": f"Bearer {token}"}

# Load previous cache if exists
if os.path.exists(PR_JSON_FILE):
    print("Cache found, no fetch.")
else:
    pr_dict = {}  # {pr_number: {title, created_at, author}}

    while True:
        response = requests.get(url, headers=headers, params=params, timeout=60)
        response.raise_for_status()
        prs = response.json()
        if not prs:
            break
        for pr in prs:
            # Only keep merged PRs (comment out to keep all)
            if pr["merged_at"] is None:
                continue

            pr_dict[str(pr["number"])] = {
                "title": pr["title"],
                "created_at": pr["created_at"],
                "author": pr["user"]["login"],
            }
        params["page"] += 1

    with open(PR_JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(pr_dict, f, ensure_ascii=False)

    print(f"Saved {len(pr_dict)} PRs to {PR_JSON_FILE}")

Saved 2591 PRs to _prs.json


In [ ]:
# Quickly clean up PRs (drop bot PRs and so on)
import json
import re

title_patterns = [
    r"(?i)pre[- ]?commit",
    r"(?i)dependabot",
]

author_patterns = [
    r"\[bot\]",
    r"(?i)pre[- ]?commit",
]

title_regexes = [re.compile(p) for p in title_patterns]
author_regexes = [re.compile(p) for p in author_patterns]


def match_any(regexes, text: str) -> bool:
    return any(r.search(text) for r in regexes)


with open(PR_JSON_FILE, encoding="utf-8") as f:
    prs = json.load(f)  # {number: {...}}

filtered = {}
uniq_dropped_titles = set()
uniq_dropped_authors = set()

for num, data in prs.items():
    title = (data.get("title") or "").strip()
    author = (data.get("author") or "").strip()

    if match_any(author_regexes, author):
        if author:
            uniq_dropped_authors.add(author)
        continue

    if match_any(title_regexes, title):
        if title:
            uniq_dropped_titles.add(title)
        continue

    filtered[num] = data

# Save cleaned
with open(PR_JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(filtered, f, indent=2, ensure_ascii=False)

print(f"Kept {len(filtered)} PRs. Saved to {PR_JSON_FILE}")
print(f"Dropped by title: (unique titles: {len(uniq_dropped_titles)})")
for t in sorted(uniq_dropped_titles):
    print("  •", (t[:120] + "…") if len(t) > 120 else t)

print(f"Dropped by author: (unique authors: {len(uniq_dropped_authors)})")
for a in sorted(uniq_dropped_authors):
    print("  •", a)

Kept 2445 PRs. Saved to _prs.json
Dropped by title: (unique titles: 0)
Dropped by author: (unique authors: 0)


In [ ]:
# Feed PR titles to the LLM year-by-year and merge JSON results.

import json
from collections import defaultdict
from openai import OpenAI

client = OpenAI()

INSTRUCTIONS = """You are an expert technical summarizer.

You will receive GitHub pull request titles for ONE YEAR.
Identify the 6-10 main technical topics for that year.

For each topic, provide a summary of under 10 words.

Rules:
- Avoid personal names and issue numbers.
- Focus on technical subject matter.
- Do not invent topics; fewer than 6 is acceptable if warranted.

Return ONLY valid JSON in exactly this structure (no extra text, no Markdown):

{
  "year": YYYY,
  "topics": ["summary_1", "summary_2", ...]
}
"""


def get_output_text(resp) -> str:
    # SDK helper: prefer .output_text if present, else fall back
    try:
        return resp.output_text
    except AttributeError:
        return resp.output[0].content[0].text


def summarize_year(year: int, titles: list[str]) -> list[dict]:
    # Build compact per-year payload
    bullets = "\n".join(f"- {t}" for t in titles)
    prompt = f"""{INSTRUCTIONS}

YEAR {year} PR titles:
{bullets}
""".strip()

    resp = client.responses.create(
        model=MODEL,
        input=prompt,
        # max_output_tokens=700,
    )
    raw = get_output_text(resp)
    data = json.loads(raw)  # expect strict JSON due to prompt

    return data["topics"]


# Load PRs
with open(PR_JSON_FILE, encoding="utf-8") as f:
    prs = json.load(f)

# Group titles by year
by_year: dict[int, list[str]] = defaultdict(list)
for _, data in prs.items():
    title = (data.get("title") or "").strip()
    if not title:
        continue
    when = (data.get("merged_at") or data.get("created_at") or "").strip()
    if len(when) < 4:
        continue
    year = int(when[:4])
    by_year[year].append(title)

# Deduplicate per-year
for y in list(by_year.keys()):
    by_year[y] = sorted(set(by_year[y]))

# Call the model year-by-year and merge
merged: dict[str, list[str]] = {}
for year in sorted(by_year):
    print(f"Working on {year=}")
    topics = summarize_year(year, by_year[year])
    print(f"{topics=}\n")
    merged[str(year)] = topics

with open(PR_TOPICS_FILE, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

Working on year=2013
topics=['Abinitio and Abinit integration and improvements', 'Molecule and structure matching enhancements', 'Bug fixes and test reliability', 'Visualization and VTK compatibility updates', 'Unit handling and improvements', 'Interface expansions to external simulation tools']

Working on year=2014
topics=['Defects and dilute solution model improvements', 'Bug fixes and code refactoring', 'Py3K compatibility updates', 'Ab initio and electronic structure enhancements', 'Surface and transformation calculations', 'Serialization and input/output parsing improvements']

Working on year=2015
topics=['Defect and solute modeling enhancements', 'Parser and input/output format improvements', 'Bug fixes and stability updates', 'Support for Gaussian and NWChem outputs', 'zeo++ and structure analysis features', 'Python 3 compatibility work']

Working on year=2016
topics=['Structure analysis methods and improvements', 'Tensor and elasticity calculation enhancements', 'Chemical env